In [1]:
import pyvista as pv
pv.set_jupyter_backend("server")

# Frame Transformations
---

In [2]:
from math import pi as PI

from python_robot.base import (
    WREF_FRAME, 
    Frame, 
    Rotation, 
    Translation, 
    Screw, 
    Vector,
    plot_frames_async,
    Axis,
    PrincipalAxis
)

## Translation

### Post-Multiplication

Create a frame {A} by translation relative to the world frame:

In [3]:
frame_A = WREF_FRAME * Translation(Vector([1, 1, 0]))
frame_A.name = "A"
print(frame_A)
print(frame_A.matrix)

Frame([x: 1, y: 1, z: 0], [alpha: 0 rad, beta: -0 rad, gamma: 0 rad])
   1         0         0         1         
   0         1         0         1         
   0         0         1         0         
   0         0         0         1         



### Pre-Multiplication

In [4]:
frame_B = Translation(Vector([1, 1, 0])) * WREF_FRAME
frame_B.name = "B"
print(frame_B)
print(frame_B.matrix)

Frame([x: 1, y: 1, z: 0], [alpha: 0 rad, beta: -0 rad, gamma: 0 rad])
   1         0         0         1         
   0         1         0         1         
   0         0         1         0         
   0         0         0         1         



> **Translation post-multiplication or pre-multiplication does not affect the result of the translation.**

## Rotation

### Post-Multiplication

Rotate frame {A} about its own Z-axis.

In [5]:
frame_C = frame_A * Rotation(Vector([0, 0, PI/4]))
frame_C.name = "C"
print(frame_C)
print(frame_C.matrix)                             

Frame([x: 1, y: 1, z: 0], [alpha: 0 rad, beta: -0 rad, gamma: 0.7854 rad])
   0.7071   -0.7071    0         1         
   0.7071    0.7071    0         1         
   0         0         1         0         
   0         0         0         1         



### Pre-Multiplication

Rotate frame {A} about the Z-axis of its reference frame, here the World Frame {W}.

In [6]:
frame_D = Rotation(Vector([0, 0, 3 * PI/4])) * frame_A
frame_D.name = "D"
print(frame_D)
print(frame_D.matrix)

Frame([x: -1.414, y: 1.11e-16, z: 0], [alpha: 0 rad, beta: -0 rad, gamma: 2.356 rad])
  -0.7071   -0.7071    0        -1.414     
   0.7071   -0.7071    0         0         
   0         0         1         0         
   0         0         0         1         



In [7]:
await plot_frames_async([frame_C, frame_D], frame_scale=0.5, show_label=True)

Widget(value='<iframe src="http://localhost:49528/index.html?ui=P_0x2707f934740_0&reconnect=auto" class="pyvis…

> **Rotation post-multiplication or pre-multiplication does affect the result of the rotation.** "Post" applies the rotation relative to the frame subject to the rotation (the rotation axis passes through the origin of the frame). "Pre" applies the rotation relative to the reference frame of the frame subject to the rotation (the rotation axis passes through the origin of the reference frame). 

### Rotation about an Arbitrary Axis

**Assignment:** *Create a frame rotated by a certain angle about the given rotation axis through the given pole.*

In [8]:
theta = PI / 2
rot_axis = PrincipalAxis.Z   # Axis.from_spherical(phi=PI/4, theta=PI/4)
pole = Vector([1, 1, 0])

The rotation axis and the pole are defined w.r.t. the world frame.

**STEP 1: Translation**<br>
First we create a frame whose origin is at the given pole. Therefore, we define a translation to move a frame initially coincident with the world
frame to the position of the pole.

In [9]:
Trans = Translation(pole)
print(Trans.matrix)

   1         0         0         1         
   0         1         0         1         
   0         0         1         0         
   0         0         0         1         



It's also interesting to look at the inverse of this translation:

In [10]:
print(Trans.inv().matrix)

   1         0         0        -1         
   0         1         0        -1         
   0         0         1         0         
   0         0         0         1         



We see that the inverse translation moves the frame back over the same x-, y-, and z-distance.

Now apply the translation relative to the world frame:

In [11]:
frame_E = WREF_FRAME * Trans
frame_E.name = "E"
print(frame_E)
print(frame_E.matrix)

Frame([x: 1, y: 1, z: 0], [alpha: 0 rad, beta: -0 rad, gamma: 0 rad])
   1         0         0         1         
   0         1         0         1         
   0         0         1         0         
   0         0         0         1         



**STEP 2: Rotation**<br>
Next, we rotate the frame about the given rotation axis by the given angle:

In [12]:
frame_F = frame_E * Rotation.about_axis(rot_axis, theta)
frame_F.name = "F"
print(frame_F)
print(frame_F.matrix)

Frame([x: 1, y: 1, z: 0], [alpha: 0 rad, beta: -0 rad, gamma: 1.571 rad])
   0        -1         0         1         
   1         0         0         1         
   0         0         1         0         
   0         0         0         1         



**STEP 3: Inverse Translation**<br>
Finally, we translate the frame back over the same x-, y-, and z-distance. Be aware however that the translation is now referring to frame {F} instead of the world frame (the translation happens along the axes of frame {F}).

In [13]:
frame_G = frame_F * Trans.inv()
frame_G.name = "G"
print(frame_G)
print(frame_G.matrix)

Frame([x: 2, y: 0, z: 0], [alpha: 0 rad, beta: -0 rad, gamma: 1.571 rad])
   0        -1         0         2         
   1         0         0         0         
   0         0         1         0         
   0         0         0         1         



Steps 1, 2, and 3 could also be done in a single step (which actually combines the three steps internally):

In [14]:
frame_H = WREF_FRAME * Rotation.about_axis(rot_axis, theta, pole)
frame_H.name = "H"
print(frame_H)
print(frame_H.matrix)

Frame([x: 2, y: 0, z: 0], [alpha: 0 rad, beta: -0 rad, gamma: 1.571 rad])
   0        -1         0         2         
   1         0         0         0         
   0         0         1         0         
   0         0         0         1         



We see that frame {H} has the same description (matrix) as frame {G}.

In [15]:
await plot_frames_async([frame_F, frame_G, frame_H], show_label=True, frame_scale=0.5)

Widget(value='<iframe src="http://localhost:49528/index.html?ui=P_0x2707fbd5ee0_1&reconnect=auto" class="pyvis…

## Transform Sequence

**Assignment:** *Consider a frame {I} and a frame {J} w.r.t. the world frame {W}. Find the description (pose) of frame {J} w.r.t. frame {I}.*

Description of frames {I} and {J} w.r.t. the world frame {W}:

In [16]:
frame_WI = Frame(origin=[-1, 2, 3], rpy_angles=[0, 0, 0], name="I")
frame_WJ = Frame(origin=[-2, 0, 0], rpy_angles=[0, 0, PI / 4], name="J")

The world frame {W} observed from frame {I}:

In [17]:
frame_IW = ~frame_WI
print(frame_IW)
print(frame_IW.matrix)

Frame([x: 1, y: -2, z: -3], [alpha: 0 rad, beta: -0 rad, gamma: 0 rad])
   1         0         0         1         
   0         1         0        -2         
   0         0         1        -3         
   0         0         0         1         



The `~`-operator inverts the frame's description (matrix), which means the "direction of view" between frames is flipped (`_WI` becomes `_IW`).

The "view arrow" from frame {I} to frame {J} can be regarded as a sequence: the view arrow from {I} to {W} followed by the view arrow from {W} to {J}:

In [18]:
frame_IJ = frame_IW * frame_WJ
print(frame_IJ)
print(frame_IJ.matrix)

Frame([x: -1, y: -2, z: -3], [alpha: 0 rad, beta: -0 rad, gamma: 0.7854 rad])
   0.7071   -0.7071    0        -1         
   0.7071    0.7071    0        -2         
   0         0         1        -3         
   0         0         0         1         



In [19]:
await plot_frames_async([frame_WI, frame_WJ], frame_scale=0.5, show_label=True)

Widget(value='<iframe src="http://localhost:49528/index.html?ui=P_0x27016b71100_2&reconnect=auto" class="pyvis…

## Screw

A screw is a combination of a translation and a rotation.

In [20]:
screw = Screw(
    axis=PrincipalAxis.Z,
    pole=Vector([1, 1, 0]),
    magnitude=PI/4,
    pitch=2
)
print(screw.matrix)

   0.7071   -0.7071    0         1         
   0.7071    0.7071    0        -0.4142    
   0         0         1         1.571     
   0         0         0         1         



The screw axis is parallel to the Z-axis of the world frame and passes through the point (1, 1, 0) in the world frame. The rotation angle about the screw axis is PI/4 rad (45°), and the pitch of the screw axis is 2 m/rad.

Create a frame {K} by applying the screw to the world frame:

In [21]:
frame_K = WREF_FRAME * screw
frame_K.name = "K"
print(frame_K)
print(frame_K.matrix)

Frame([x: 1, y: -0.4142, z: 1.571], [alpha: 0 rad, beta: -0 rad, gamma: 0.7854 rad])
   0.7071   -0.7071    0         1         
   0.7071    0.7071    0        -0.4142    
   0         0         1         1.571     
   0         0         0         1         



In [22]:
await plot_frames_async([frame_K], frame_scale=0.5, show_label=True)

Widget(value='<iframe src="http://localhost:49528/index.html?ui=P_0x27016b72d20_3&reconnect=auto" class="pyvis…

A screw can also be either purely rotational or purely translational. The screw is purely rotational when the pitch is zero (or `None`). When the pole is `None`, the screw is purely translational. In this case, parameter `magnitude` specifies the translational distance in the direction of the screw axis, while parameter `pitch` is unused. 

### Rotational Screw

In [23]:
rot_screw = Screw(
    axis=PrincipalAxis.Z,
    pole=Vector([1, 1, 0]),
    magnitude=3 * PI/4
)

In [24]:
frame_L = WREF_FRAME * rot_screw
frame_L.name = "L"
print(frame_L)
print(frame_L.matrix)

Frame([x: 2.414, y: 1, z: 0], [alpha: 0 rad, beta: -0 rad, gamma: 2.356 rad])
  -0.7071   -0.7071    0         2.414     
   0.7071   -0.7071    0         1         
   0         0         1         0         
   0         0         0         1         



In [25]:
await plot_frames_async([frame_L], frame_scale=0.5, show_label=True)

Widget(value='<iframe src="http://localhost:49528/index.html?ui=P_0x2702381c170_4&reconnect=auto" class="pyvis…

### Translational Screw

In [26]:
trans_screw = Screw(
    axis=Axis.from_spherical(phi=PI/4, theta=PI/2),
    pole=None,
    magnitude=2
)

In [27]:
frame_M = WREF_FRAME * trans_screw
frame_M.name = "M"
print(frame_M)
print(frame_M.matrix)

Frame([x: 1.414, y: 1.414, z: 1.225e-16], [alpha: 0 rad, beta: -0 rad, gamma: 0 rad])
   1         0         0         1.414     
   0         1         0         1.414     
   0         0         1         0         
   0         0         0         1         



In [28]:
await plot_frames_async([frame_M], frame_scale=0.5, show_label=True)

Widget(value='<iframe src="http://localhost:49528/index.html?ui=P_0x2702381c0e0_5&reconnect=auto" class="pyvis…